# Day 5: Data Leakage in Machine Learning

This notebook demonstrates different types of data leakage using
a used-car price prediction dataset.

Goal:
- Understand what data leakage is
- Identify common leakage types
- Learn how to prevent them


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression


In [3]:
# Load dataset
df = pd.read_csv("RAW_DATA\\carprices.csv")
df.head()


,Mileage,Age(yrs),Sell Price($)
0,69000,6,18000
1,35000,3,34000
2,57000,5,26100
3,22500,2,40000
4,46000,4,31500


## Dataset Overview

Features:
- Mileage: total distance driven
- Age(yrs): age of the car in years

Target:
- Sell Price($): price of the used car

At prediction time, only Mileage and Age are available.


In [4]:
X = df[['Mileage', 'Age(yrs)']]
y = df['Sell Price($)']

## 1. Train–Test Leakage

Train–test leakage happens when the same data is used
both for training and evaluation.

To avoid this, we always split data into separate
training and testing sets.


In [18]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=42)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)


Training size: (14, 2)
Testing size: (6, 2)


## 2. Preprocessing Leakage

Preprocessing leakage occurs when transformations
(scaling, normalization, imputation) are fitted
on the full dataset before splitting.

This allows test data to influence training.


In [7]:
# ❌ WRONG: Scaling before train-test split
scaler = StandardScaler()
X_scaled_wrong = scaler.fit_transform(X)


❌ This is leakage because the scaler learns statistics
(mean and standard deviation) from the entire dataset,
including test data.


In [8]:
# ✅ CORRECT: Fit only on training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


✅ This prevents preprocessing leakage because
test data is never used to compute scaling parameters.


## 3. Target Leakage

Target leakage happens when a feature contains
information derived from the target variable.

Rule:
If a feature would not be available at prediction time,
it must not be used.


In [9]:
# ❌ Example of target leakage
df["price_per_km"] = df["Sell Price($)"] / df["Mileage"]

df[['Mileage', 'price_per_km', 'Sell Price($)']].head()


,Mileage,price_per_km,Sell Price($)
0,69000,0.260870,18000
1,35000,0.971429,34000
2,57000,0.457895,26100
3,22500,1.777778,40000
4,46000,0.684783,31500


❌ This is leakage because "price_per_km" uses the target
(Sell Price) to predict itself.

## 4. Temporal Leakage

Temporal leakage occurs when future data is used
to predict past events.

This usually happens in time-based datasets.


In [19]:
# ❌ WRONG: Random split on time-based data
df.sample(frac=0.8, random_state=42)


,Mileage,Age(yrs),Sell Price($),price_per_km
0,69000,6,18000,0.260870
17,69000,5,19700,0.285507
15,25400,3,35000,1.377953
1,35000,3,34000,0.971429
8,91000,8,12000,0.131868
5,59000,5,26750,0.453390
11,79000,7,19500,0.246835
3,22500,2,40000,1.777778
18,87600,8,12800,0.146119
16,28000,2,35500,1.267857


❌ Random sampling breaks time order.

✅ Correct approach:
- Sort by time
- Train on past data
- Test on future data


## 5. Group Leakage

Group leakage happens when related records
appear in both training and testing sets.

Examples:
- Same customer
- Same car
- Same user


In [12]:
from sklearn.model_selection import GroupShuffleSplit

# Example (not used here because dataset has no groups)
# gss = GroupShuffleSplit(test_size=0.3)


Rule:
All records belonging to the same group must
exist entirely in either train or test.


## Training Model Without Leakage


In [15]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)


LinearRegression()

In [16]:
score = model.score(X_test_scaled, y_test)
score


0.8911199230718784

## Summary: Data Leakage

Types of leakage covered:
1. Train–test leakage
2. Preprocessing leakage
3. Target leakage
4. Temporal leakage
5. Group leakage

Key rules followed:
- Split data before preprocessing
- Fit transformations on training data only
- Avoid target-derived features
- Respect time order
- Prevent group overlap

This approach mirrors real-world ML production pipelines.
